In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Sangram Mishra\AppData\Local\Temp\ipykernel_3672\1202363267.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
#fist we ahve to load the documnets
loader = PyPDFLoader("attn.pdf")
docs = loader.load()
print("the loaded length of doc is ",len(docs)) 

the loaded length of doc is  11


In [5]:
spliter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)
chunks = spliter.split_documents(docs)
print("the created chunk length is ", len(chunks))

the created chunk length is  76


In [7]:
embedings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 941.54it/s]


In [8]:
vector_store = Chroma.from_documents(
    embedding=embedings,
    documents=chunks,
    persist_directory="db/chroma"
)

In [9]:
retriver = vector_store.as_retriever(
    search_kwargs={
        "k":3
    }
)

In [32]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model = "qwen3:8b")

In [11]:
llm.invoke("what is capital of india")

AIMessage(content='The capital of India is **New\u202fDelhi**.', additional_kwargs={}, response_metadata={'model': 'nemotron-3-nano:4b', 'created_at': '2026-09-27T17:44:06.2445823Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12259829100, 'load_duration': 11383364000, 'prompt_eval_count': 21, 'prompt_eval_duration': 145087000, 'eval_count': 35, 'eval_duration': 716001000, 'logprobs': None, 'model_name': 'nemotron-3-nano:4b', 'model_provider': 'ollama'}, id='lc_run--01a0e3f7-5b76-7ba2-bea0-1576291693fe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 35, 'total_tokens': 56})

In [33]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}
"""
)

In [34]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [35]:
rag_chain = (
    {"context":retriver|format_docs,"question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

In [39]:
qustion = "can you brief compleate ateention mechanism in 500 words in most easay manner"
answer = rag_chain.invoke(qustion)
print("qustion: ", qustion)
print("answer: ", answer)

qustion:  can you brief compleate ateention mechanism in 500 words in most easay manner
answer:  **Attention Mechanism: A Simple Explanation**  

The attention mechanism is a powerful tool in artificial intelligence, particularly in natural language processing (NLP), that helps models focus on the most relevant parts of input data. Imagine you’re reading a sentence, and your brain automatically highlights the words that matter most for understanding the meaning. The attention mechanism mimics this process by allowing models to weigh different parts of the input differently, improving their ability to capture context and relationships.  

There are different types of attention mechanisms, but one of the most popular is **self-attention** (also called **intra-attention**). Self-attention is used when a model needs to understand a single sequence of data, like a sentence. Here’s how it works:  

When a model processes a sentence, it breaks it into individual words or tokens (e.g., "The ca